# PDF Extractor for Google Colab

Parses a **true (text-based) PDF** and produces a ZIP bundle of structured Markdown files.

**Pipeline**

1. **Choose PDF** - upload it or point to a path / Drive file.
2. **Choose pages** - `all`, `1,3,5`, `2-7`, or any mix.
3. **Layout analysis** - [Docling](https://github.com/docling-project/docling) (optionally the [Granite-Docling 258M](https://huggingface.co/ibm-granite/granite-docling-258M) VLM) produces:
   - `_layout/layout.md`
   - `_layout/layout.json` (lossless `DoclingDocument`)
4. **Data extraction** - the JSON is walked in reading order; for every element we go back to the original PDF and extract the raw data with **PyMuPDF** (text + cropped images) and **Camelot Stream** (tables).
5. **Bundle** - the output folder is zipped and offered for download. The bundle contains:
   - `00_contents.md` - table of contents (one row per element)
   - `01_main.md` - plain text of the PDF with placeholders pointing at sidecars
   - `tables/table_NNN.md` - one Markdown file per table
   - `images/image_NNN.png` + `.md` sidecar - one per picture
   - `code/`, `formulas/` - one file each (when present)
   - `layout/` - a copy of the Docling `.md` + lossless `.json`

> Run the cells **top-to-bottom**. Each step prompts you with `input()` when needed.

## 0. Install dependencies

Camelot needs **Ghostscript** at the OS level; the rest are pure pip installs.
The first install run takes ~2-3 minutes on Colab.

In [ ]:
import sys, subprocess, importlib

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# OS-level deps for Camelot (Ghostscript) - only on Colab/Linux.
try:
    subprocess.check_call(
        ["apt-get", "-qq", "install", "-y", "ghostscript", "python3-tk"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
except Exception:
    pass  # non-Linux: assume user already has Ghostscript or skip Camelot

for pkg, mod in [
    ("pymupdf",       "pymupdf"),
    ("docling",       "docling"),
    ("camelot-py[base]", "camelot"),
    ("opencv-python-headless", "cv2"),
    ("pandas",        "pandas"),
    ("Pillow",        "PIL"),
]:
    try:
        importlib.import_module(mod)
    except ImportError:
        print(f"installing {pkg} ...")
        _pip(pkg)

print("All dependencies ready.")

## 1. Load the pipeline module

The full pipeline lives in `pdf_extractor.py` (defined inline below so this notebook is self-contained on Colab).

In [ ]:
%%writefile pdf_extractor.py
"""
pdf_extractor.py
================

Parse a true (text-based) PDF and extract its content into a structured
folder of Markdown files, using:

    * Docling                  -> layout analysis + lossless DoclingDocument JSON
    * PyMuPDF (`pymupdf`)      -> plain text, image cropping, page geometry
    * Camelot (Stream flavor)  -> table re-extraction guided by Docling bboxes
"""

from __future__ import annotations

import json
import re
import shutil
import sys
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pymupdf

try:
    import camelot  # type: ignore
    HAS_CAMELOT = True
except Exception:
    HAS_CAMELOT = False

from docling.document_converter import DocumentConverter


# ---------------------------------------------------------------------------
# User input helpers
# ---------------------------------------------------------------------------

def parse_page_spec(spec: str, total: int) -> List[int]:
    """Parse 'all' | '1,3,5' | '2-7' | mixes -> sorted 1-indexed page list."""
    s = (spec or "").strip().lower()
    if not s or s in ("all", "*"):
        return list(range(1, total + 1))
    pages: set = set()
    for token in s.split(","):
        token = token.strip()
        if not token:
            continue
        if "-" in token:
            a, b = token.split("-", 1)
            try:
                ai, bi = int(a.strip()), int(b.strip())
            except ValueError:
                continue
            for p in range(min(ai, bi), max(ai, bi) + 1):
                if 1 <= p <= total:
                    pages.add(p)
        else:
            try:
                p = int(token)
            except ValueError:
                continue
            if 1 <= p <= total:
                pages.add(p)
    return sorted(pages)


def make_subset_pdf(src_path, pages: List[int], dst_path) -> List[int]:
    src = pymupdf.open(str(src_path))
    dst = pymupdf.open()
    try:
        for p in pages:
            dst.insert_pdf(src, from_page=p - 1, to_page=p - 1)
        dst.save(str(dst_path))
    finally:
        dst.close()
        src.close()
    return list(pages)


def slugify(text: str, maxlen: int = 40) -> str:
    text = re.sub(r"[^\w\-]+", "_", (text or "").strip()).strip("_")
    return (text[:maxlen] or "item").lower()


# ---------------------------------------------------------------------------
# Coordinate / bbox helpers
# ---------------------------------------------------------------------------

def bbox_to_pymu_rect(bbox: dict, page_height: float) -> pymupdf.Rect:
    l, t, r, b = float(bbox["l"]), float(bbox["t"]), float(bbox["r"]), float(bbox["b"])
    origin = (bbox.get("coord_origin") or "BOTTOMLEFT").upper()
    if origin == "BOTTOMLEFT":
        return pymupdf.Rect(l, page_height - t, r, page_height - b)
    return pymupdf.Rect(l, t, r, b)


def bbox_to_camelot_area(bbox: dict, page_height: float) -> str:
    l, t, r, b = float(bbox["l"]), float(bbox["t"]), float(bbox["r"]), float(bbox["b"])
    origin = (bbox.get("coord_origin") or "BOTTOMLEFT").upper()
    if origin == "BOTTOMLEFT":
        return f"{l},{t},{r},{b}"
    return f"{l},{page_height - t},{r},{page_height - b}"


# ---------------------------------------------------------------------------
# Table helpers
# ---------------------------------------------------------------------------

def docling_table_to_markdown(table_data: dict) -> str:
    rows: List[List[str]] = []
    grid = table_data.get("grid")
    if grid:
        for row in grid:
            out_row = []
            for cell in row:
                txt = (cell.get("text") or "")
                out_row.append(txt.replace("|", "\\|").replace("\n", " ").strip())
            rows.append(out_row)
    else:
        cells = table_data.get("table_cells") or []
        nrows = int(table_data.get("num_rows") or 0)
        ncols = int(table_data.get("num_cols") or 0)
        if nrows and ncols:
            rows = [["" for _ in range(ncols)] for _ in range(nrows)]
            for c in cells:
                i = int(c.get("start_row_offset_idx", 0))
                j = int(c.get("start_col_offset_idx", 0))
                if 0 <= i < nrows and 0 <= j < ncols:
                    txt = (c.get("text") or "")
                    rows[i][j] = txt.replace("|", "\\|").replace("\n", " ").strip()
    if not rows:
        return ""
    width = max(len(r) for r in rows)
    rows = [r + [""] * (width - len(r)) for r in rows]
    header, body = rows[0], rows[1:]
    out = ["| " + " | ".join(header) + " |",
           "| " + " | ".join(["---"] * width) + " |"]
    for r in body:
        out.append("| " + " | ".join(r) + " |")
    return "\n".join(out)


def dataframe_to_markdown(df) -> str:
    cols = [str(c).replace("|", "\\|") for c in df.columns]
    out = ["| " + " | ".join(cols) + " |",
           "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in df.iterrows():
        cells = ["" if v is None else str(v).replace("|", "\\|").replace("\n", " ").strip()
                 for v in row.tolist()]
        out.append("| " + " | ".join(cells) + " |")
    return "\n".join(out)


def extract_table_with_camelot(pdf_path: str, page_in_subset: int, area: str) -> Optional[str]:
    if not HAS_CAMELOT:
        return None
    try:
        tables = camelot.read_pdf(
            pdf_path, pages=str(page_in_subset),
            flavor="stream", table_areas=[area],
            suppress_stdout=True,
        )
        if tables.n == 0:
            return None
        df = tables[0].df
        df.columns = [f"col_{i}" if not str(c).strip() else str(c) for i, c in enumerate(df.iloc[0])]
        df = df.iloc[1:].reset_index(drop=True)
        return dataframe_to_markdown(df)
    except Exception:
        return None


# ---------------------------------------------------------------------------
# Layout analysis
# ---------------------------------------------------------------------------

@dataclass
class LayoutResult:
    subset_pdf: Path
    page_map: List[int]
    layout_md: Path
    layout_json: Path
    docling_dict: Dict[str, Any]


def run_layout(src_pdf, pages: List[int], work_root: Path, use_granite: bool = False) -> LayoutResult:
    work_root.mkdir(parents=True, exist_ok=True)
    layout_dir = work_root / "_layout"
    layout_dir.mkdir(parents=True, exist_ok=True)

    subset_pdf = work_root / "_subset.pdf"
    page_map = make_subset_pdf(src_pdf, pages, subset_pdf)

    if use_granite:
        from docling.datamodel.base_models import InputFormat
        from docling.datamodel.pipeline_options import VlmPipelineOptions
        from docling.datamodel import vlm_model_specs
        from docling.document_converter import PdfFormatOption
        from docling.pipeline.vlm_pipeline import VlmPipeline

        opts = VlmPipelineOptions(vlm_options=vlm_model_specs.GRANITEDOCLING_TRANSFORMERS)
        converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_cls=VlmPipeline, pipeline_options=opts
                )
            }
        )
    else:
        converter = DocumentConverter()

    result = converter.convert(str(subset_pdf))
    doc = result.document

    layout_md = layout_dir / "layout.md"
    layout_json = layout_dir / "layout.json"
    doc.save_as_markdown(layout_md)
    doc_dict = doc.export_to_dict()
    layout_json.write_text(json.dumps(doc_dict, ensure_ascii=False, indent=2), encoding="utf-8")

    return LayoutResult(
        subset_pdf=subset_pdf, page_map=page_map,
        layout_md=layout_md, layout_json=layout_json,
        docling_dict=doc_dict,
    )


# ---------------------------------------------------------------------------
# Walk DoclingDocument & extract data
# ---------------------------------------------------------------------------

@dataclass
class ContentItem:
    idx: int
    kind: str
    label: str
    page: int
    preview: str
    sidecar: Optional[str] = None


def _resolve_ref(doc: dict, ref: str) -> Optional[dict]:
    if not ref or not ref.startswith("#/"):
        return None
    parts = ref[2:].split("/")
    cur: Any = doc
    for p in parts:
        if isinstance(cur, list):
            try:
                cur = cur[int(p)]
            except (ValueError, IndexError):
                return None
        elif isinstance(cur, dict):
            cur = cur.get(p)
            if cur is None:
                return None
        else:
            return None
    return cur if isinstance(cur, dict) else None


def _iter_reading_order(doc: dict):
    body = doc.get("body") or {}
    children = body.get("children") or []
    for child in children:
        ref = child.get("$ref") or child.get("cref")
        if not ref:
            continue
        item = _resolve_ref(doc, ref)
        if item is not None:
            yield ref, item


def _item_kind(ref: str, item: dict) -> Tuple[str, str]:
    label = (item.get("label") or "").lower()
    section = ref.split("/")[1] if ref.startswith("#/") else ""
    if section == "tables":
        return "table", label or "table"
    if section == "pictures":
        return "picture", label or "picture"
    if label in {"section_header", "title", "page_header", "subtitle_level_1"}:
        return "heading", label
    if label in {"list_item"}:
        return "list", label
    if label in {"caption"}:
        return "caption", label
    if label in {"code"}:
        return "code", label
    if label in {"formula"}:
        return "formula", label
    return "text", label or "paragraph"


def _heading_level(label: str) -> int:
    if label == "title":
        return 1
    if label == "section_header":
        return 2
    return 3


def _prov_page_and_bbox(item: dict) -> Tuple[Optional[int], Optional[dict]]:
    prov = item.get("prov") or []
    if not prov:
        return None, None
    p0 = prov[0]
    return p0.get("page_no"), p0.get("bbox")


def _picture_caption(doc: dict, item: dict) -> str:
    caps = []
    for c in item.get("captions") or []:
        ref = c.get("$ref") or c.get("cref")
        cap = _resolve_ref(doc, ref) if ref else None
        if cap and cap.get("text"):
            caps.append(cap["text"])
    return " ".join(caps).strip()


def extract_to_folder(src_pdf, layout: LayoutResult, out_root: Path, image_dpi: int = 200) -> Path:
    out_root.mkdir(parents=True, exist_ok=True)
    tables_dir = out_root / "tables"
    images_dir = out_root / "images"
    codes_dir = out_root / "code"
    formulas_dir = out_root / "formulas"
    layout_copy_dir = out_root / "layout"
    for d in (tables_dir, images_dir, codes_dir, formulas_dir, layout_copy_dir):
        d.mkdir(parents=True, exist_ok=True)

    shutil.copy2(layout.layout_md, layout_copy_dir / "layout.md")
    shutil.copy2(layout.layout_json, layout_copy_dir / "layout.json")

    doc = layout.docling_dict
    page_map = layout.page_map

    src_doc = pymupdf.open(str(src_pdf))
    sub_doc = pymupdf.open(str(layout.subset_pdf))

    contents: List[ContentItem] = []
    main_lines: List[str] = []
    current_orig_page: Optional[int] = None
    table_no = image_no = code_no = formula_no = 0
    item_idx = 0

    try:
        for ref, item in _iter_reading_order(doc):
            item_idx += 1
            kind, label = _item_kind(ref, item)
            page_in_subset, bbox = _prov_page_and_bbox(item)
            if page_in_subset is None or not (1 <= page_in_subset <= len(page_map)):
                continue
            orig_page = page_map[page_in_subset - 1]

            if orig_page != current_orig_page:
                if current_orig_page is not None:
                    main_lines.append("")
                main_lines.append(f"---\n\n## Page {orig_page}\n")
                current_orig_page = orig_page

            sub_page = sub_doc[page_in_subset - 1]
            src_page = src_doc[orig_page - 1]
            page_h_sub = sub_page.rect.height

            if kind in ("text", "heading", "list", "caption"):
                pdf_text = ""
                if bbox:
                    rect = bbox_to_pymu_rect(bbox, page_h_sub)
                    try:
                        pdf_text = src_page.get_text("text", clip=rect).strip()
                    except Exception:
                        pdf_text = ""
                if not pdf_text:
                    pdf_text = (item.get("text") or "").strip()
                if not pdf_text:
                    continue

                if kind == "heading":
                    level = _heading_level(label)
                    main_lines.append(f"\n{'#' * level} {pdf_text}\n")
                    preview = pdf_text
                elif kind == "list":
                    main_lines.append(f"- {pdf_text}")
                    preview = pdf_text
                elif kind == "caption":
                    main_lines.append(f"\n*{pdf_text}*\n")
                    preview = pdf_text
                else:
                    main_lines.append(pdf_text + "\n")
                    preview = pdf_text

                contents.append(ContentItem(
                    idx=item_idx, kind=kind, label=label,
                    page=orig_page, preview=preview[:120],
                ))

            elif kind == "table":
                table_no += 1
                name = f"table_{table_no:03d}"
                rel_path = f"tables/{name}.md"
                md_table = None
                if bbox:
                    area = bbox_to_camelot_area(bbox, page_h_sub)
                    md_table = extract_table_with_camelot(str(layout.subset_pdf), page_in_subset, area)
                if not md_table:
                    md_table = docling_table_to_markdown(item.get("data") or {})

                caption = _picture_caption(doc, item)
                bbox_str = json.dumps(bbox) if bbox else "n/a"
                body = (
                    f"# Table {table_no:03d}\n\n"
                    f"- **Original page:** {orig_page}\n"
                    f"- **BBox (subset PDF):** {bbox_str}\n"
                    + (f"- **Caption:** {caption}\n" if caption else "")
                    + f"\n{md_table or '_(empty table)_'}\n"
                )
                (tables_dir / f"{name}.md").write_text(body, encoding="utf-8")

                placeholder = f"\n> **[TABLE {table_no:03d}]** -> [`{rel_path}`]({rel_path}){' - ' + caption if caption else ''}\n"
                main_lines.append(placeholder)
                contents.append(ContentItem(
                    idx=item_idx, kind="table", label=label, page=orig_page,
                    preview=(caption or f"Table {table_no:03d}")[:120], sidecar=rel_path,
                ))

            elif kind == "picture":
                image_no += 1
                name = f"image_{image_no:03d}"
                png_rel = f"images/{name}.png"
                md_rel = f"images/{name}.md"

                if bbox:
                    rect = bbox_to_pymu_rect(bbox, page_h_sub)
                    try:
                        pix = src_page.get_pixmap(clip=rect, dpi=image_dpi, alpha=False)
                        pix.save(str(images_dir / f"{name}.png"))
                    except Exception:
                        pix = src_page.get_pixmap(dpi=image_dpi, alpha=False)
                        pix.save(str(images_dir / f"{name}.png"))

                caption = _picture_caption(doc, item)
                bbox_str = json.dumps(bbox) if bbox else "n/a"
                md_body = (
                    f"# Image {image_no:03d}\n\n"
                    f"- **Original page:** {orig_page}\n"
                    f"- **BBox (subset PDF):** {bbox_str}\n"
                    + (f"- **Caption:** {caption}\n" if caption else "")
                    + f"\n![{name}]({name}.png)\n"
                )
                (images_dir / f"{name}.md").write_text(md_body, encoding="utf-8")

                placeholder = (
                    f"\n> **[IMAGE {image_no:03d}]** -> [`{png_rel}`]({png_rel}) "
                    f"| sidecar: [`{md_rel}`]({md_rel})"
                    + (f" - {caption}" if caption else "") + "\n"
                )
                main_lines.append(placeholder)
                contents.append(ContentItem(
                    idx=item_idx, kind="picture", label=label, page=orig_page,
                    preview=(caption or f"Image {image_no:03d}")[:120], sidecar=png_rel,
                ))

            elif kind == "code":
                code_no += 1
                name = f"code_{code_no:03d}"
                rel_path = f"code/{name}.md"
                code_text = (item.get("text") or "").rstrip()
                if bbox and not code_text:
                    rect = bbox_to_pymu_rect(bbox, page_h_sub)
                    try:
                        code_text = src_page.get_text("text", clip=rect).rstrip()
                    except Exception:
                        pass
                lang = (item.get("code_language") or "").strip()
                fence = f"```{lang}\n{code_text}\n```"
                (codes_dir / f"{name}.md").write_text(
                    f"# Code {code_no:03d}\n\n- **Original page:** {orig_page}\n\n{fence}\n",
                    encoding="utf-8",
                )
                main_lines.append(f"\n{fence}\n")
                contents.append(ContentItem(
                    idx=item_idx, kind="code", label=label, page=orig_page,
                    preview=(code_text.splitlines()[0] if code_text else "")[:120],
                    sidecar=rel_path,
                ))

            elif kind == "formula":
                formula_no += 1
                name = f"formula_{formula_no:03d}"
                rel_path = f"formulas/{name}.md"
                latex = (item.get("text") or "").strip()
                block = f"$$\n{latex}\n$$" if latex else "_(empty formula)_"
                (formulas_dir / f"{name}.md").write_text(
                    f"# Formula {formula_no:03d}\n\n- **Original page:** {orig_page}\n\n{block}\n",
                    encoding="utf-8",
                )
                main_lines.append(f"\n{block}\n")
                contents.append(ContentItem(
                    idx=item_idx, kind="formula", label=label, page=orig_page,
                    preview=latex[:120], sidecar=rel_path,
                ))
    finally:
        src_doc.close()
        sub_doc.close()

    main_path = out_root / "01_main.md"
    src_name = Path(str(src_pdf)).name
    header = (
        f"# {src_name}\n\n"
        f"_Extracted with Docling + PyMuPDF + Camelot. "
        f"Tables/images/etc. are referenced as sidecar files._\n"
    )
    main_path.write_text(header + "\n".join(main_lines).rstrip() + "\n", encoding="utf-8")

    toc_lines = [
        "# Document Contents", "",
        f"Source: `{src_name}`  ",
        f"Total elements: **{len(contents)}**  ",
        f"Tables: **{table_no}** | Images: **{image_no}** | "
        f"Code blocks: **{code_no}** | Formulas: **{formula_no}**",
        "",
        "| # | Type | Page | Preview / File |",
        "|---|------|------|----------------|",
    ]
    for c in contents:
        if c.sidecar:
            ref = f"[`{c.sidecar}`]({c.sidecar})"
            if c.preview:
                ref = f"{c.preview} - {ref}"
        else:
            ref = c.preview.replace("|", "\\|") if c.preview else ""
        toc_lines.append(f"| {c.idx} | {c.kind} | {c.page} | {ref} |")
    (out_root / "00_contents.md").write_text("\n".join(toc_lines) + "\n", encoding="utf-8")
    return out_root


# ---------------------------------------------------------------------------
# Packaging
# ---------------------------------------------------------------------------

def zip_folder(folder: Path, zip_path: Path) -> Path:
    folder = Path(folder)
    zip_path = Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in folder.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=path.relative_to(folder.parent))
    return zip_path


def run(pdf_path, page_spec: str = "all", out_dir="output",
        use_granite: bool = False, image_dpi: int = 200) -> Path:
    pdf_path = Path(pdf_path).expanduser().resolve()
    if not pdf_path.is_file():
        raise FileNotFoundError(pdf_path)
    out_dir = Path(out_dir).expanduser().resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    with pymupdf.open(str(pdf_path)) as src:
        total_pages = src.page_count
    pages = parse_page_spec(page_spec, total_pages)
    if not pages:
        raise ValueError(f"No valid pages in spec={page_spec!r} (PDF has {total_pages} pages)")

    bundle_name = f"{pdf_path.stem}_pages_{pages[0]}-{pages[-1]}"
    work_root = out_dir / bundle_name
    if work_root.exists():
        shutil.rmtree(work_root)
    work_root.mkdir(parents=True, exist_ok=True)

    print(f"[1/3] Layout analysis on {len(pages)} page(s) (Docling{' + Granite VLM' if use_granite else ''})...")
    layout = run_layout(pdf_path, pages, work_root, use_granite=use_granite)
    print(f"      -> {layout.layout_md.relative_to(out_dir)}")
    print(f"      -> {layout.layout_json.relative_to(out_dir)}")

    print("[2/3] Extracting text / tables / images from PDF using JSON map...")
    extracted_dir = work_root / "extracted"
    extract_to_folder(pdf_path, layout, extracted_dir, image_dpi=image_dpi)
    n_files = sum(1 for _ in extracted_dir.rglob("*") if _.is_file())
    print(f"      -> {extracted_dir.relative_to(out_dir)} ({n_files} files)")

    print("[3/3] Zipping bundle...")
    zip_path = out_dir / f"{bundle_name}.zip"
    zip_folder(extracted_dir, zip_path)
    print(f"      -> {zip_path}")
    return zip_path


In [ ]:
import importlib, pdf_extractor
importlib.reload(pdf_extractor)
from pdf_extractor import (
    parse_page_spec, run_layout, extract_to_folder, zip_folder, run,
    HAS_CAMELOT,
)
print("pdf_extractor loaded from:", pdf_extractor.__file__)
print("Camelot available:", HAS_CAMELOT)

## 2. Choose the PDF

You can:
- **upload** a file from your computer (default if running on Colab), or
- type a **path** (works for Google Drive once you mount it with `from google.colab import drive; drive.mount('/content/drive')`), or
- paste a **direct https URL** to a PDF.

In [ ]:
import os, shutil
from pathlib import Path
from urllib.request import urlretrieve

PDF_PATH = None

def _on_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

print("How do you want to provide the PDF?")
print("  [1] Upload from this computer (Colab only)")
print("  [2] Type a path on the runtime (e.g. /content/file.pdf or /content/drive/MyDrive/...)")
print("  [3] Paste an https URL")
choice = input("Choice [1/2/3] (default 1 on Colab, else 2): ").strip()
if not choice:
    choice = "1" if _on_colab() else "2"

if choice == "1":
    if not _on_colab():
        raise RuntimeError("Upload widget only works in Google Colab. Use option 2 or 3 instead.")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    PDF_PATH = Path(list(uploaded.keys())[0]).resolve()

elif choice == "2":
    p = input("Enter the PDF path: ").strip().strip('"').strip("'")
    PDF_PATH = Path(p).expanduser().resolve()

elif choice == "3":
    url = input("PDF URL: ").strip()
    dst = Path("/content/downloaded.pdf") if _on_colab() else Path("downloaded.pdf").resolve()
    print(f"Downloading {url} -> {dst} ...")
    urlretrieve(url, dst)
    PDF_PATH = dst.resolve()

else:
    raise ValueError(f"Unknown choice: {choice!r}")

assert PDF_PATH and PDF_PATH.is_file(), f"PDF not found: {PDF_PATH}"
print(f"\nPDF: {PDF_PATH}  ({PDF_PATH.stat().st_size/1024:.1f} KB)")

## 3. Choose pages

Examples: `all`, `1`, `1,3,5`, `2-7`, `1, 3-5, 9`.

In [ ]:
import pymupdf

with pymupdf.open(str(PDF_PATH)) as _doc:
    TOTAL_PAGES = _doc.page_count
print(f"The PDF has {TOTAL_PAGES} page(s).")

PAGE_SPEC = input(f"Which pages? (default 'all'): ").strip() or "all"
PAGES = parse_page_spec(PAGE_SPEC, TOTAL_PAGES)
if not PAGES:
    raise ValueError(f"No valid pages parsed from {PAGE_SPEC!r}")
print(f"Will process {len(PAGES)} page(s): {PAGES if len(PAGES) <= 20 else f'{PAGES[:10]} ... {PAGES[-3:]}'}")

USE_GRANITE = input("Use Granite-Docling VLM (slower, needs GPU)? [y/N]: ").strip().lower().startswith("y")
print("Use Granite-Docling VLM:", USE_GRANITE)

## 4. Layout analysis -> `layout.md` + `layout.json`

Docling runs over the selected pages and we save **both** the markdown view (human readable) and the lossless `DoclingDocument` JSON (machine readable, used as the map for the next step).

In [ ]:
OUT_DIR = Path("/content/output" if _on_colab() else "output").resolve()
BUNDLE_NAME = f"{PDF_PATH.stem}_pages_{PAGES[0]}-{PAGES[-1]}"
WORK_ROOT = OUT_DIR / BUNDLE_NAME

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Running Docling{' + Granite VLM' if USE_GRANITE else ''} on {len(PAGES)} page(s) ...")
LAYOUT = run_layout(PDF_PATH, PAGES, WORK_ROOT, use_granite=USE_GRANITE)
print("Saved:")
print("  ", LAYOUT.layout_md)
print("  ", LAYOUT.layout_json)

print("\n----- preview of layout.md (first 800 chars) -----")
print(LAYOUT.layout_md.read_text(encoding="utf-8")[:800])

## 5. Data extraction from the PDF (using the JSON map)

Walk the lossless JSON in reading order; for every element, go back to the **original PDF** and pull:

- **Text** with `PyMuPDF` (`get_text(clip=bbox)`),
- **Tables** with **Camelot Stream** (using the Docling bbox as `table_areas`); falls back to Docling's parsed table when Camelot returns nothing,
- **Pictures** by cropping a pixmap at the bbox.

Each table / image / code block / formula gets its own sidecar `.md`. The main file references them inline.

In [ ]:
EXTRACTED_DIR = WORK_ROOT / "extracted"
extract_to_folder(PDF_PATH, LAYOUT, EXTRACTED_DIR, image_dpi=200)

print("Generated files:")
for p in sorted(EXTRACTED_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(EXTRACTED_DIR), f"({p.stat().st_size} bytes)")

## 6. Zip & download

In [ ]:
ZIP_PATH = OUT_DIR / f"{BUNDLE_NAME}.zip"
zip_folder(EXTRACTED_DIR, ZIP_PATH)
print(f"Bundle ready: {ZIP_PATH}  ({ZIP_PATH.stat().st_size/1024:.1f} KB)")

if _on_colab():
    from google.colab import files
    files.download(str(ZIP_PATH))
else:
    print("\n(Not on Colab) Open the file manually from the path above.")

### Optional: one-shot run

If you'd rather skip the interactive cells and run the full pipeline in one call (great for batching), use:

```python
zip_path = run(
    pdf_path="/content/your.pdf",
    page_spec="1-5",           # or "all" / "1,3,7"
    out_dir="/content/output",
    use_granite=False,         # True to use Granite-Docling VLM
    image_dpi=200,
)
print(zip_path)
```